In [23]:
import sys
sys.path.insert(0, "/Users/nautilus/cobblestone_study")

import pandas as pd
import holidays
from config import CLEAN_PARQUET, PEAK_HOURS, PEAK_WEEKDAYS

df = pd.read_parquet("/Users/nautilus/cobblestone_study/data/clean.parquet")
df.head()

,prices,load_fc,wind_fc,solar_fc,actual_load,actual_gen,residual_load_fc
2024-06-12 00:00:00+02:00,88.85,43510.4775,10215.6250,0.00,45384.5300,39235.5900,33294.8525
2024-06-12 01:00:00+02:00,85.35,42501.1275,10056.7950,0.00,43697.0700,38589.1400,32444.3325
2024-06-12 02:00:00+02:00,80.29,42080.3675,9831.2350,0.00,43011.9625,39225.4450,32249.1325
2024-06-12 03:00:00+02:00,82.11,42441.8100,9400.2350,0.00,43002.1425,38758.1925,33041.5750
2024-06-12 04:00:00+02:00,86.11,44073.2475,8982.6375,1.16,44276.2100,38214.9025,35089.4500


In [24]:
de_holidays = holidays.Germany()

df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
# df["is_holiday"] = 
df["is_holiday"] = pd.Series(df.index.date, index=df.index).map(lambda d: d in de_holidays).astype(int)
df["is_peak"] = (
    df["hour"].isin(PEAK_HOURS) & df["dayofweek"].isin(PEAK_WEEKDAYS)
).astype(int)

df[["hour", "dayofweek", "month", "is_holiday", "is_peak"]].head(10)

,hour,dayofweek,month,is_holiday,is_peak
2024-06-12 00:00:00+02:00,0,2,6,0,0
2024-06-12 01:00:00+02:00,1,2,6,0,0
2024-06-12 02:00:00+02:00,2,2,6,0,0
2024-06-12 03:00:00+02:00,3,2,6,0,0
2024-06-12 04:00:00+02:00,4,2,6,0,0
2024-06-12 05:00:00+02:00,5,2,6,0,0
2024-06-12 06:00:00+02:00,6,2,6,0,0
2024-06-12 07:00:00+02:00,7,2,6,0,0
2024-06-12 08:00:00+02:00,8,2,6,0,1
2024-06-12 09:00:00+02:00,9,2,6,0,1


In [25]:
# Lagged Price
df["price_lag_24"] = df["prices"].shift(24) # Same hour yesterday
df["price_lag_168"] = df["prices"].shift(168)  # same hour last week

df[["prices", "price_lag_24", "price_lag_168"]].head(30)

,prices,price_lag_24,price_lag_168
2024-06-12 00:00:00+02:00,88.85,NaN,NaN
2024-06-12 01:00:00+02:00,85.35,NaN,NaN
2024-06-12 02:00:00+02:00,80.29,NaN,NaN
2024-06-12 03:00:00+02:00,82.11,NaN,NaN
2024-06-12 04:00:00+02:00,86.11,NaN,NaN
2024-06-12 05:00:00+02:00,94.65,NaN,NaN
2024-06-12 06:00:00+02:00,124.74,NaN,NaN
2024-06-12 07:00:00+02:00,136.40,NaN,NaN
2024-06-12 08:00:00+02:00,114.62,NaN,NaN
2024-06-12 09:00:00+02:00,78.65,NaN,NaN


In [27]:
df["price_roll_mean_24"] = df["prices"].shift(1).rolling(24).mean()
df["price_roll_std_24"] = df["prices"].shift(1).rolling(24).std()
df["price_roll_mean_168"] = df["prices"].shift(1).rolling(168).mean()
# shift(1) needed to prevent look-ahead bias
# Since without this, it includes its own value 
# hour 1 include hour 1's own price! but we don't know yet at prediction time

In [29]:
FEATURE_COLS = [
    "load_fc", "wind_fc", "solar_fc", "residual_load_fc",
    "hour", "dayofweek", "month", "is_holiday", "is_peak",
    "price_lag_24", "price_lag_168",
    "price_roll_mean_24", "price_roll_std_24", "price_roll_mean_168",
]

df_clean = df.dropna(subset=FEATURE_COLS + ["prices"])

print(f"Rows before dropna: {len(df)}")
print(f"Rows after dropna:  {len(df_clean)}")
df_clean[FEATURE_COLS].head()


Rows before dropna: 17521
Rows after dropna:  17181


,load_fc,wind_fc,solar_fc,residual_load_fc,hour,dayofweek,month,is_holiday,is_peak,price_lag_24,price_lag_168,price_roll_mean_24,price_roll_std_24,price_roll_mean_168
2024-06-19 00:00:00+02:00,43340.4575,5483.8325,0.0000,37856.6250,0,2,6,0,0,96.92,88.85,96.198333,32.646993,69.135714
2024-06-19 01:00:00+02:00,42049.5275,5347.7225,0.0000,36701.8050,1,2,6,0,0,90.29,85.35,96.324583,32.655762,69.201786
2024-06-19 02:00:00+02:00,41669.4800,5391.2150,0.0000,36278.2650,2,2,6,0,0,87.53,80.29,96.372083,32.647431,69.237976
2024-06-19 03:00:00+02:00,41919.0025,5599.7575,0.0000,36319.2450,3,2,6,0,0,89.29,82.11,96.343750,32.655732,69.277024
2024-06-19 04:00:00+02:00,43329.4675,5813.9575,52.3225,37463.1875,4,2,6,0,0,90.29,86.11,96.197500,32.696531,69.298869
